# DiTy CE-only + stack (fine-tuned model from prior OOF run)

1. Score train/test with fine-tuned `DiTy` CE
2. CatBoost YetiRank on lexical + text + CE
3. Write `submission.csv` (stack) and `submission_ce_only.csv`

In [ ]:
import os, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch

OUT = Path('/kaggle/working')
MODELS = OUT / 'models'
OUT.mkdir(exist_ok=True); MODELS.mkdir(exist_ok=True)

hits = list(Path('/kaggle/input').rglob('train.csv'))
DATA = hits[0].parent
CODE = Path('/kaggle/input/amml-ranking-2026-code')
CE = Path('/kaggle/input/amml-ranking-2026-dity-ce-ft/ce_st_mmarco')
if not CE.exists():
    CE = next(Path('/kaggle/input').rglob('model.safetensors')).parent
print('DATA', DATA, 'CE', CE)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rank-bm25==0.2.2', 'sentence-transformers==3.3.1', 'datasets'])
sys.path.insert(0, str(CODE))
try:
    from src.ce_train import oof_st_cross_encoder_scores
except Exception:
    repo = Path('/tmp/amml')
    if not repo.exists():
        subprocess.check_call(['git','clone','--depth','1','https://github.com/VenistorAlishi/amml-ranking-competition-2026.git', str(repo)])
    sys.path.insert(0, str(repo))

from src.preprocessing import prepare_frame
from src.features import assemble_features, CAT_FEATURES
from src.metrics import mean_ndcg_at_k
from src.train import FEATURE_EXCLUDE, default_params
from sentence_transformers import CrossEncoder
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupKFold

train = prepare_frame(pd.read_csv(DATA/'train.csv'))
test = prepare_frame(pd.read_csv(DATA/'test.csv'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', device, torch.cuda.get_device_name(0) if device=='cuda' else '')

# Fresh honest OOF DiTy (strong) — ignore cached FT if we have GPU time
DO_FRESH_OOF = True
if DO_FRESH_OOF:
    from src.ce_train import oof_st_cross_encoder_scores
    ce_oof, ce_te = oof_st_cross_encoder_scores(
        train, test, models_dir=MODELS,
        model_name='DiTy/cross-encoder-russian-msmarco',
        epochs=3, batch_size=16, max_len=512, n_splits=5,
    )
else:
    model = CrossEncoder(str(CE), max_length=512, device=device)
    def score(df, bs=64):
        pairs=[[str(q), f'{t} {x}'.strip()] for q,t,x in zip(df['query'],df['title'],df['text'])]
        return np.asarray(model.predict(pairs, batch_size=bs, show_progress_bar=True), dtype=float)
    print('scoring cached FT model...')
    ce_oof = score(train); ce_te = score(test)

print('CE-only OOF', mean_ndcg_at_k(train[['id','query_id','relevance']].assign(prediction=ce_oof)))
pd.DataFrame({'id': test['id'], 'prediction': ce_te}).to_csv(OUT/'submission_ce_only.csv', index=False)

print('features...')
feat_tr, feat_te = assemble_features(train, test)
for feat, raw, sc in ((feat_tr, train, ce_oof), (feat_te, test, ce_te)):
    feat['ce_score'] = sc
    g = feat.groupby('query_id')['ce_score']
    feat['ce_score_qrank'] = g.rank(method='average', ascending=False)
    mean = g.transform('mean'); std = g.transform('std').replace(0, np.nan)
    feat['ce_score_qz'] = ((feat['ce_score']-mean)/std).fillna(0.0)
    feat['query_text'] = raw['query'].astype(str).to_numpy()
    feat['title_text'] = raw['title'].fillna('').astype(str).to_numpy()
    feat['passage_text'] = raw['text'].fillna('').astype(str).to_numpy()

TEXT=['query_text','title_text','passage_text']
def cols(df): return [c for c in df.columns if c not in FEATURE_EXCLUDE]
def pool(df, c, label=True):
    g=df['query_id'].astype('category').cat.codes.to_numpy(); o=np.argsort(g, kind='mergesort')
    cat=[c.index(x) for x in CAT_FEATURES if x in c]; tx=[c.index(x) for x in TEXT if x in c]
    kw=dict(cat_features=cat, text_features=tx, group_id=g[o])
    X=df.iloc[o][c]
    return (Pool(X, df.iloc[o]['relevance'].to_numpy(), **kw) if label else (Pool(X, **kw), o))

params=default_params(); params.update(dict(iterations=2500, learning_rate=0.03, depth=8, border_count=128, od_wait=200))
df=feat_tr.copy(); df['relevance']=train['relevance'].to_numpy(); c=cols(df)
gkf=GroupKFold(5); oof=np.zeros(len(df)); scores=[]
for fold,(tr_i,va_i) in enumerate(gkf.split(df, df['relevance'], groups=df['query_id'])):
    tr=df.iloc[tr_i].reset_index(drop=True); va=df.iloc[va_i].reset_index(drop=True)
    vg=va['query_id'].astype('category').cat.codes.to_numpy(); vo=np.argsort(vg, kind='mergesort'); vs=va.iloc[vo]
    cat=[c.index(x) for x in CAT_FEATURES if x in c]; tx=[c.index(x) for x in TEXT if x in c]
    m=CatBoostRanker(**params)
    m.fit(pool(tr,c,True), eval_set=Pool(vs[c], vs['relevance'].to_numpy(), group_id=vg[vo], cat_features=cat, text_features=tx), use_best_model=True)
    pred=np.empty(len(va)); pred[vo]=m.predict(Pool(vs[c], group_id=vg[vo], cat_features=cat, text_features=tx))
    oof[va_i]=pred; nd=mean_ndcg_at_k(va.assign(prediction=pred)); scores.append(nd)
    print(f'fold {fold} NDCG={nd:.6f} iter={m.get_best_iteration()}')
print('CV', float(np.mean(scores)), 'CE-OOF', mean_ndcg_at_k(train[['id','query_id','relevance']].assign(prediction=ce_oof)))

# Honest OOF blend weight CE vs CB
from sklearn.preprocessing import MinMaxScaler
def qmm(qid, vals):
    out=np.zeros(len(vals)); df=pd.DataFrame({'q':qid,'v':vals})
    for _,idx in df.groupby('q').groups.items():
        idx=list(idx); s=df.loc[idx,'v'].to_numpy().reshape(-1,1)
        out[idx]=MinMaxScaler().fit_transform(s).ravel()
    return out
cb_n=qmm(train['query_id'], oof); ce_n=qmm(train['query_id'], ce_oof)
best_w,best_nd=1.0,-1
for w in np.linspace(0,1,21):
    nd=mean_ndcg_at_k(train[['id','query_id','relevance']].assign(prediction=w*cb_n+(1-w)*ce_n))
    if nd>best_nd: best_nd,best_w=nd,float(w)
print('blend w_cb', best_w, 'ndcg', best_nd)

fp={**params,'iterations':2000}; fp.pop('od_type',None); fp.pop('od_wait',None)
m=CatBoostRanker(**fp); m.fit(pool(df,c,True))
tep,teo=pool(feat_te,c,False); pred=np.empty(len(feat_te)); pred[teo]=m.predict(tep)
cb_te=qmm(test['query_id'], pred); ce_te_n=qmm(test['query_id'], ce_te)
blend=best_w*cb_te+(1-best_w)*ce_te_n
pd.DataFrame({'id':test['id'],'prediction':pred}).to_csv(OUT/'submission.csv', index=False)
pd.DataFrame({'id':test['id'],'prediction':blend}).to_csv(OUT/'submission_blend.csv', index=False)
pd.DataFrame({'id':test['id'],'prediction':ce_te}).to_csv(OUT/'submission_ce_only.csv', index=False)
print('wrote submission.csv / blend / ce_only')
